hw1: Metatranscriptomic analysis of bronchoalveolar lavage fluid sample in order to identify the causative agent of infection

In [1]:
quast ~/Desktop/spades_scaffolds.fasta -o ~/Desktop/quast_out

SyntaxError: invalid syntax (2291109626.py, line 1)

| Assembly | spades_scaffolds |
|----------|------------------|
| contigs (>= 0 bp) | 111219 |
| contigs (>= 1000 bp) | 666 |
| contigs (>= 5000 bp) | 4 |
| contigs (>= 10000 bp) | 1 |
| contigs (>= 25000 bp) | 1 |
| contigs (>= 50000 bp) | 0 |
| Total length (>= 0 bp) | 27362468 |
| Total length (>= 1000 bp) | 1042805 |
| Total length (>= 5000 bp) | 48741 |
| Total length (>= 10000 bp) | 29907 |
| Total length (>= 25000 bp) | 29907 |
| Total length (>= 50000 bp) | 0 |
| contigs | 3667 |
| Largest contig | 29907 |
| Total length | 2988329 |
| GC (%) | 45.92 |
| N50 | 779 |
| N90 | 532 |
| auN | 1351.8 |
| L50 | 1182 |
| L90 | 3087 |
| N's per 100 kbp | 17.37 |

The command allows you to run the quast program on our data. The `-o` flag allows you to output the result of the program to the output directory `~/Desktop/quast_out`.

In [ ]:
samtools faidx spades_scaffolds.fasta

With this command, we can index the contigs in `spades_scaffolds.fasta` and see their main metrics after the team's work - coverage, length

In [ ]:
more spades_scaffolds.fasta.fai  | awk '$2 >= 3000 {print $1}' | xargs samtools faidx spades_scaffolds.fasta > 3000.fa

| Node | Length (bp) | Position | Line Bases | Line Width | Coverage |
|------|-------------|----------|------------|------------|----------|
| NODE_1 | 29 907 | 36 | 60 | 61 | 150.82 |
| NODE_2 | 8 134 | 30 475 | 60 | 61 | 6.18 |
| NODE_3 | 5 584 | 38 782 | 60 | 61 | 13 759.61 |
| NODE_4 | 5 116 | 44 494 | 60 | 61 | 18.12 |
| NODE_5 | 4 814 | 49 730 | 60 | 61 | 12.30 |
| NODE_6 | 4 266 | 54 659 | 60 | 61 | 43.85 |
| NODE_7 | 4 074 | 59 030 | 60 | 61 | 7.19 |
| NODE_8 | 3 958 | 63 205 | 60 | 61 | 4.04 |
| NODE_9 | 3 842 | 67 263 | 60 | 61 | 27.24 |
| NODE_10 | 3 800 | 71 204 | 60 | 61 | 8.26 |
| NODE_11 | 3 791 | 75 102 | 60 | 61 | 9.90 |
| NODE_12 | 3 766 | 78 991 | 60 | 61 | 9.20 |
| NODE_13 | 3 721 | 82 855 | 60 | 61 | 21.06 |
| NODE_14 | 3 711 | 86 673 | 60 | 61 | 8.32 |
| NODE_15 | 3 618 | 90 480 | 60 | 61 | 6.27 |
| NODE_16 | 3 609 | 94 194 | 60 | 61 | 10.85 |
| NODE_17 | 3 494 | 97 899 | 60 | 61 | 18.37 |
| NODE_18 | 3 476 | 101 487 | 60 | 61 | 26.72 |
| NODE_19 | 3 468 | 105 055 | 60 | 61 | 6.43 |
| NODE_20 | 3 346 | 108 615 | 60 | 61 | 4.91 |
| NODE_21 | 3 341 | 112 052 | 60 | 61 | 35.07 |
| NODE_22 | 3 172 | 115 483 | 60 | 61 | 5.14 |
| NODE_23 | 3 129 | 118 742 | 60 | 61 | 6.23 |

This command filters and extracts long contigs from the assembly. First, `more spades_scaffolds.fasta.fai` displays the index file containing contig names and their lengths. Then `awk '$2 >= 3000 {print $1}'` selects only those contigs where the second column (length) is 3000 base pairs or greater, printing their names. Finally, `xargs samtools faidx spades_scaffolds.fasta` takes these contig names and extracts the corresponding sequences from the original FASTA file, saving them into `3000.fa`. Using this command, we select contigs with a length above 3000 bp, which is important for finding the sequence we are interested in, as longer contigs are more likely to contain complete or near-complete genes and provide more reliable annotation results.

In [ ]:
samtools faidx 3000.fa "NODE_1_length_29907_cov_150.822528" > NODE_1.fa

This command extracts a specific contig from the filtered assembly file. `samtools faidx 3000.fa` accesses the indexed FASTA file containing only the long contigs, and `"NODE_1_length_29907_cov_150.822528"` specifies the exact contig name to retrieve. The extracted sequence is then saved to `NODE_1.fa` using the redirection operator `>`. After selecting the sequence we are interested in, we move the selected contig to a separate file for further genomic annotation, allowing us to work with a single contig of interest rather than the entire assembly.

In [ ]:
prokka NODE_1.fa     --kingdom Viruses     --outdir prokka_NODE_1     --prefix NODE_1_virus     --compliant     --cpus 4     --locustag VIRUS_001

This command allows you to run an annotation of the selected sequence. Flags are necessary for: 

`-- kingdom` viruses have special features in the organization of the genome, so Prokka uses special models

`--compliant` — to meet the requirements of international databases

`--cpus 4` — speeds up processing, especially for large genomes

`--locustag` — provides unique and understandable names for genes

In [ ]:
samtools faidx NODE_1.fa "NODE_1_length_29907_cov_150.822528:4519-8340" | seqkit seq -r -p -t dna > spike_gene.fast

Using this command, we can select a specific section of the virus genome (namely, in positions 4519 to 8340), and output this information to a new file in the output directory.

`-r` — creates a reverse complementary sequence (reverse complement)

`-p`  — makes a complementary sequence (usually used together with -r)

`-t`  — indicates that the sequence is DNA (not RNA or protein)